# C3 riêng của Nhật Huy — kiểm hồ sơ huấn luyện và metric lịch sử
    **Giới hạn đã xác nhận:** workspace có log, metadata, confusion counts và bảng kết quả,
    nhưng **thiếu ba bundle checkpoint/scores C3 riêng**. Notebook này thực sự chạy các
    phép kiểm hồ sơ và tính lại bảy metric từ TP/FP/FN/TN; **không suy luận model C3 riêng**.
    Không thay bundle đó bằng DistilBERT của benchmark chung. C3 chung đã được chạy
    lại đủ ba seed trong notebook 04 và có protocol khác.

    Nghiên cứu riêng này chỉ dùng train/validation; các số dưới không phải test.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/notebook_support.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.notebook_support import artifacts_root
from src.datasets.goemotions import load_goemotions, multi_hot, sha256, REVISION
from src.evaluation.protocols import read_json
from src.evaluation.metrics import evaluate_multilabel
from src.models.transformer import METRIC_NAMES
ARTIFACTS = artifacts_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_colwidth', 100)
print('Python:', sys.version.split()[0])
print('Dataset revision:', REVISION)
print('Artifacts:', ARTIFACTS.name)

REPORT = ARTIFACTS/'reports/c3_distilbert/full'
protocol = read_json(REPORT/'protocol.json')
frames, names, manifest = load_goemotions(ARTIFACTS, write_metadata=False, splits=('train','validation'))
print('Chế độ lần chạy: kiểm hồ sơ lịch sử và confusion counts, không nạp checkpoint riêng.')

## 1. Cấu hình, dataset và phạm vi đã khóa

In [ ]:
config = protocol['config']
assert config == read_json(ROOT/'configs/distilbert_full.json')
assert protocol['seeds'] == [42,123,2026] and not protocol['test_used']
assert len(frames['train']) == 43410 and len(frames['validation']) == 5426
assert not (set(frames['train'].id) & set(frames['validation'].id))
display(pd.Series(config))
display(pd.DataFrame([dict(split=s, rows=len(df), unique_ids=df.id.nunique()) for s,df in frames.items()]))

## 2. Kiểm rõ tệp checkpoint/scores còn thiếu

In [ ]:
availability = []
for seed in protocol['seeds']:
    run = ARTIFACTS/f'data/processed/c3_distilbert/full/seed_{seed}'
    availability.append(dict(seed=seed, checkpoint_available=(run/'best/model.safetensors').is_file(),
        scores_available=(run/'validation_scores.npz').is_file(),
        historical_metadata_available=(REPORT/f'run_seed_{seed}.json').is_file()))
display(pd.DataFrame(availability))
print('Không chứng nhận reload, loss forward hoặc suy luận C3 riêng mới khi bundle chưa có.')

## 3. Metadata huấn luyện đã bàn giao và best epoch

In [ ]:
history = pd.read_csv(REPORT/'history_all_seeds.csv')
receipts = []
for seed in protocol['seeds']:
    meta = read_json(REPORT/f'run_seed_{seed}.json')
    rows = history[history.seed == seed]
    best = rows.sort_values(['validation_macro_f1','epoch'], ascending=[False,True]).iloc[0]
    assert meta['best_epoch'] == int(best.epoch) and not meta['test_used']
    assert meta['n_train'] == 43410 and meta['n_validation'] == 5426
    receipts.append(dict(seed=seed, best_epoch=meta['best_epoch'], n_train=meta['n_train'],
        n_validation=meta['n_validation'], gpu=meta['environment']['gpu'],
        model_sha256=meta['artifact_sha256']['best/model.safetensors']))
display(pd.DataFrame(receipts))

## 4. Lịch sử học và tài nguyên — đọc từ log cũ

In [ ]:
import matplotlib.pyplot as plt
display(history)
display(pd.read_csv(REPORT/'resources_all_seeds.csv'))
fig, axes = plt.subplots(1,2,figsize=(10,3.5))
for seed, group in history.groupby('seed'):
    axes[0].plot(group.epoch, group.train_loss, marker='o', label=f'seed {seed}')
    axes[1].plot(group.epoch, group.validation_macro_f1, marker='o', label=f'seed {seed}')
axes[0].set_title('Train loss lịch sử'); axes[1].set_title('Validation Macro-F1 lịch sử')
for ax in axes: ax.set_xlabel('Epoch đã chạy'); ax.legend()
plt.tight_layout(); plt.show(); plt.close()
for seed in protocol['seeds']:
    print('Log seed',seed,':',*(REPORT/f'train_seed_{seed}.log').read_text(encoding='utf-8').splitlines()[-3:],sep='\n')

## 5. Tính lại đủ bảy metric từ confusion counts thật
    TP/FP/FN/TN của đủ 28 nhãn được giữ trong CSV bàn giao. Các phép tính này không
    cần weights; chúng kiểm tính nhất quán của số liệu lịch sử, không kiểm lại scores
    hoặc checkpoint bị thiếu. Số mẫu mỗi nhãn phải đúng 5.426.

In [ ]:
from src.notebook_support import metrics_from_confusion
counts = pd.read_csv(REPORT/'per_label_all_seeds.csv')
reported = pd.read_csv(REPORT/'threshold_comparison_runs.csv')
support = multi_hot(frames['train'].labels.tolist()).sum(axis=0)
rows, differences = [], []
for (seed,mode), group in counts.groupby(['seed','mode'], sort=True):
    group = group.sort_values('label_id')
    assert group.label.tolist() == names and group.label_id.tolist() == list(range(28))
    np.testing.assert_array_equal(group.train_support, support)
    metric = metrics_from_confusion(group, 5426)
    expected = reported[(reported.seed == seed) & (reported['mode'] == mode)].iloc[0]
    filename = ('fixed' if mode == 'fixed_0.5' else 'tuned')
    original = read_json(REPORT/f'metrics_{filename}_seed_{seed}.json')
    for key in METRIC_NAMES:
        differences.extend([abs(metric[key]-expected[key]),abs(metric[key]-original[key])])
    rows.append(dict(seed=seed, mode=mode, **metric))
assert len(rows) == 6 and max(differences) <= 1e-12
recalculated = pd.DataFrame(rows)
display(recalculated)
print('PASS: 6 dòng validation; sai lệch từ confusion counts:',max(differences))

## 6. Mean và sample std từ đủ ba seed của nghiên cứu riêng

In [ ]:
summary = recalculated.groupby('mode')[list(METRIC_NAMES)].agg(['mean','std'])
display(summary)
saved = pd.read_csv(REPORT/'threshold_comparison_mean_std.csv')
for _,row in saved.iterrows():
    assert abs(summary.loc[row['mode'],(row['metric'],'mean')]-row['mean']) <= 1e-12
    assert abs(summary.loc[row['mode'],(row['metric'],'std')]-row['sample_std']) <= 1e-12
print('PASS: mean và sample std khớp CSV lịch sử. Không gộp các run này vào C3 chung.')

## 7. Ngưỡng riêng và nhãn hiếm — validation đã dùng chọn ngưỡng

In [ ]:
threshold_rows=[]
for seed in protocol['seeds']:
    saved=read_json(REPORT/f'thresholds_seed_{seed}.json')
    assert saved['label_names']==names and not saved['test_used']
    for label,value in zip(names,saved['thresholds']):
        threshold_rows.append(dict(seed=seed,label=label,threshold=value))
display(pd.DataFrame(threshold_rows).pivot(index='label',columns='seed',values='threshold').reindex(names))
display(pd.read_csv(REPORT/'rare_labels_mean_std.csv'))
print('Tuned-validation không phải test độc lập; không chứng minh cải thiện trên test.')

## 8. Kiểm nguyên văn, ID và nhãn thật của ví dụ lỗi đã bàn giao

In [ ]:
examples=read_json(REPORT/'error_examples.json')
by_id=frames['validation'].set_index('id')
for example in examples:
    row=by_id.loc[example['id']]
    assert row.text==example['text'] and [names[j] for j in row.labels]==example['true_labels']
display(pd.DataFrame([{k:v for k,v in e.items() if k!='scores'} for e in examples]).groupby('group',sort=False).head(3))
display(pd.read_csv(REPORT/'error_group_counts.csv'))
print('PASS:',len(examples),'ví dụ có ID/văn bản/nhãn thật khớp snapshot validation.')

## 9. Hồ sơ kiểm demo cũ và bước để khôi phục suy luận
    File verification dưới được tạo trên máy chạy C3 riêng trước đây. Ngày kiểm và
    kết quả đó được giữ nguyên; không diễn giải thành lượt kiểm UI mới hôm nay.
    Muốn chạy model C3 riêng cần nhận lại `best/`, `run.json`, scores/ID và các hash.
    Sau khi có bundle, dùng `scripts.distilbert.verify_distilbert`; demo cuối nhóm
    hiện dùng BERT seed 123 tại `app.py`.

In [ ]:
receipt=read_json(REPORT/'verification.json')
display(pd.Series({k:receipt[k] for k in ('checked_at_utc','status','run','test_used')}))
display(pd.DataFrame(receipt['checks'].items(),columns=['Kiểm lịch sử','Kết quả đã ghi']))
print('Hoàn tất kiểm hồ sơ C3 riêng; checkpoint mới nạp: 0; suy luận mới: 0.')
print('C3 CHUNG: xem notebook 04 với ba checkpoint đã suy luận test thật.')